# 🛰️ Week 6 – Satellite Land-Use Classification with EuroSAT
**Project 2: Deep Learning Applications (Individual)** · Introduction to Applied AI · BS 8th Semester · Department of Electronics

**Name:** ________________  **Roll No:** ________________  **Date:** ________________

---
## 1. Problem statement
**Task.** Classify 64 × 64 Sentinel-2 satellite tiles into **10 land-use / land-cover classes** (AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture, PermanentCrop, Residential, River, SeaLake) using a fine-tuned **ResNet18**.

**Why it matters.** Land-use and crop maps are the base layer for flood-risk planning, crop monitoring, and urban-growth tracking. Manual surveys are slow and expensive, so automatic classification of satellite tiles can update these maps much faster.

**Scope note (honest framing).** EuroSAT has **no flood class**. The *River* and *SeaLake* classes are the water classes that a flood-mapping pipeline would build on, so I treat water-class recall as the flood-relevant result. I do **not** claim this model detects floods.

| Item | Choice |
|---|---|
| Domain | Satellite imagery: land use, crops, water (flood-risk context) |
| Dataset | EuroSAT (Sentinel-2 RGB), about 27,000 labelled images, 10 classes, Europe (Helber et al.) |
| **Key metric** | **Per-class recall** (plus macro-F1 for model selection). A missed class is a hole in the map, and accuracy can hide it. |
| **Known pitfall** | **Region and season shift.** All tiles are European and from a limited set of acquisition conditions. A model trained here may degrade on other regions, crop calendars, haze, or sensors. |
| Backbone | ResNet18 (ImageNet pre-trained), two-phase fine-tuning |

## 2. Notebook map
| Part | Content |
|---|---|
| 0 | Configuration and imports |
| 1 | Index and audit the dataset |
| 2 | Leakage audit: exact and near-duplicates, group split |
| 3 | Transforms and imbalance handling |
| 4 | Build, train, fine-tune; select by macro-F1 |
| 5 | Test once; per-class recall; water-class threshold; season/haze stress test |
| 6 | Grad-CAM shortcut check |
| 7 | Save, README, LinkedIn post, Check Your Understanding |

# Part -1: Colab Setup (2 min)
**Before running anything:** go to **Runtime → Change runtime type → T4 GPU → Save.** Then run the two cells below. The first checks the GPU, the second downloads the EuroSAT RGB dataset (about 90 MB, 27,000 images) with torchvision. Colab keeps files only for the current session, so the notebook downloads the data again each time you reconnect.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')

In [ ]:
from torchvision.datasets import EuroSAT
_ = EuroSAT(root='/content/data', download=True)     # RGB version, 27,000 images, 10 classes
print('Download finished.')

# Part 0: Configuration Cell (5 min)
This is the only cell that changes between datasets. `find_class_root()` searches the downloaded data folder for the folder that holds the 10 EuroSAT class folders (RGB `.jpg`/`.png` images), so a wrong path does not stop the notebook. **Check the printed path** before continuing.

In [ ]:
import os

def find_class_root(base='/content/data'):
    """Find the folder that directly contains the EuroSAT class folders (RGB images)."""
    for dp, dn, fn in os.walk(base):
        if {'Forest', 'River', 'SeaLake'}.issubset(set(dn)):
            sample_dir = os.path.join(dp, 'Forest')
            files = os.listdir(sample_dir)
            if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
                return dp
    return None

# ============ CHANGE ONLY THIS CELL FOR YOUR OWN DATASET ============
DATA_ROOT = find_class_root() or '/content/data/eurosat/2750'
TRAIN_DIR = DATA_ROOT          # folder with one subfolder per class
TEST_DIR = None                # EuroSAT has no test set: the code carves a grouped test set
GRAYSCALE = False              # RGB satellite tiles
IMG_SIZE = 224                 # 64x64 tiles are upsampled to 224 for the ImageNet backbone
BACKBONE = 'resnet18'          # or 'resnet50', 'efficientnet_b0'
GROUP_PATTERN = None           # EuroSAT file names (e.g. Forest_1234.jpg) carry no scene id
POSITIVE = None                # multi-class task: no single positive class
TARGET_RECALL = None           # multi-class task
# ======================================================================
print('DATA_ROOT =', DATA_ROOT)
print('Class folders:', sorted(d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))))

In [ ]:
import os
import re
import time
import copy
import hashlib
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
import torchvision.transforms.functional as TF
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score,
                             roc_auc_score, roc_curve)
from IPython.display import Markdown, display

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

R = {}   # results dictionary: every key number is stored here and reused in the README and LinkedIn post

### Configuration choices and why
- **`GRAYSCALE = False`**: satellite RGB tiles carry colour that separates classes (water, vegetation, built-up). Converting to grey would throw away that evidence.
- **`IMG_SIZE = 224`**: the ImageNet-pretrained backbone expects about 224 px input. The 64 × 64 tiles are upsampled, which adds no new detail, so I do not expect a gain from larger sizes.
- **`TEST_DIR = None`**: the dataset ships as one folder per class with no official test split, so I carve a stratified test set from the data myself.
- **`GROUP_PATTERN = None`**: file names contain only the class and a running number, so there is no scene or patient id. Every image is its own group. I discuss the consequence for leakage in Part 2.
- **`POSITIVE = None`**: this is a 10-class problem, so the binary threshold cell is replaced by per-class recall and a one-vs-rest analysis of a flood-relevant water class (Part 5).
- **`BACKBONE = 'resnet18'`**: small, fast on a Colab T4 GPU, and a strong baseline for 64 px tiles. A larger backbone is only worth trying after the audit is clean.

# Part 1: Index and Audit Your Dataset (10 min)
### Task 1.1: One row per image

In [ ]:
def index_folder(folder):
    """One row per image: path, class label, file name, group id."""
    rows = []
    for cls in sorted(os.listdir(folder)):
        cdir = os.path.join(folder, cls)
        if not os.path.isdir(cdir):
            continue
        for f in sorted(os.listdir(cdir)):
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif')):
                rows.append({'path': os.path.join(cdir, f), 'label': cls,
                             'file': f})
    df = pd.DataFrame(rows)
    if GROUP_PATTERN:
        m = df['file'].str.extract(GROUP_PATTERN, expand=False)
        df['group'] = m.fillna(df['file'])      # no match: its own group
    else:
        df['group'] = df['file']
    return df

train_df = index_folder(TRAIN_DIR)
test_df = index_folder(TEST_DIR) if TEST_DIR else None
classes = sorted(train_df['label'].unique())

counts = train_df['label'].value_counts().reindex(classes)
print(counts.to_string())
print(f'Total images: {len(train_df)}  |  classes: {len(classes)}')
print(f'Imbalance ratio (max / min): {counts.max() / counts.min():.1f}')
print(f'Images per group: {len(train_df) / train_df.group.nunique():.2f}')

R['n_images'] = int(len(train_df)); R['k'] = len(classes)
R['counts'] = {c: int(counts[c]) for c in classes}
R['ratio'] = float(counts.max() / counts.min())
R['imgs_per_group'] = float(len(train_df) / train_df.group.nunique())

ax = counts.plot(kind='bar', figsize=(9, 3.5), color='#00838f')
ax.set_title('Images per class (full dataset)')
ax.set_ylabel('images')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Task 1.2: What do the images look like?

In [ ]:
sample = train_df.sample(min(300, len(train_df)), random_state=0)
info = []
for p in sample['path']:
    with Image.open(p) as im:
        info.append({'w': im.size[0], 'h': im.size[1], 'mode': im.mode})
info = pd.DataFrame(info)
print(info.describe().round(0).loc[['min', '50%', 'max']])
print('Colour modes:', info['mode'].value_counts().to_dict())
R['size_min'] = (int(info.w.min()), int(info.h.min()))
R['size_max'] = (int(info.w.max()), int(info.h.max()))
R['modes'] = info['mode'].value_counts().to_dict()

fig, axes = plt.subplots(len(classes), 5, figsize=(11, 2.2 * len(classes)))
axes = np.atleast_2d(axes)
for r, c in enumerate(classes):
    paths = train_df[train_df.label == c]['path'].sample(5, random_state=1)
    for ax, p in zip(axes[r], paths):
        ax.imshow(Image.open(p).convert('RGB'))
        ax.set_title(c, fontsize=8)
        ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Shortcut hunting: do classes differ in plain colour / brightness (a cue that is NOT land use)?
rows = []
for c in classes:
    sub = train_df[train_df.label == c]['path']
    for p in sub.sample(min(60, len(sub)), random_state=0):
        a = np.asarray(Image.open(p).convert('RGB'), dtype=np.float32)
        rows.append({'class': c, 'R': a[..., 0].mean(), 'G': a[..., 1].mean(),
                     'B': a[..., 2].mean(), 'brightness': a.mean()})
col = pd.DataFrame(rows).groupby('class').mean().round(1)
print(col.to_string())
R['color_table'] = col
col[['R', 'G', 'B']].plot(kind='bar', figsize=(9, 3.5), color=['#c62828', '#2e7d32', '#1565c0'])
plt.title('Mean R, G, B per class (60 sampled tiles each)')
plt.ylabel('mean pixel value (0-255)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## ✍️ Write it: Part 1 (Dataset audit)
*(Replace every `___` with the number from your own output above. Edit the wording so it reflects what **you** see in your grid.)*

**1. Class counts and imbalance.** The dataset has **___ images in 10 classes** (table and bar chart above). The largest class is **___** (___ images) and the smallest is **___** (___ images), so the **imbalance ratio is ___**. A ratio near 1.5 is *mild* imbalance, not the severe imbalance of the chest X-ray reference set. I still handle it (Part 3) and select models by **macro-F1**, because accuracy rewards the large classes and can hide a weak small one such as Pasture.

**2. Colour mode and size.** All sampled images are **RGB** (mode ___) and **___ × ___ pixels**, so there are no mixed sizes or greyscale files to fix. Each tile covers a very small ground area, so fine detail is limited, and I upsample to 224 × 224 only to match the pretrained backbone.

**3. First shortcut suspect.** The mean-colour table and the sample grid show that classes differ strongly in **overall colour and brightness**. For example, *Forest* tiles are dark green, *SeaLake* and *River* tiles are dark blue or blue-green, and *Industrial* / *Residential* tiles are brighter and greyer. Colour is partly real evidence (water *is* blue), but it is also a **seasonal and sensor-dependent cue**: the same crop field looks different in spring and autumn, and haze shifts colours. My first shortcut suspect is therefore **global colour cast / brightness instead of texture and shape**. I test it in Part 5 (season/haze stress test) and Part 6 (Grad-CAM and masking).

# Part 2: Leakage Audit (10 min)
Leakage means information from the test set reaching training, which makes the test score look better than reality. Two checks: **exact duplicates** (identical files) and **near-duplicates** (the same scene resized, recompressed, or brightened). Because EuroSAT has no test folder, I first check exact duplicates, then create the split, and then run the near-duplicate search **between the splits I created**.
### Task 2.1: Exact duplicates (MD5)

In [ ]:
def md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

train_df['md5'] = [md5(p) for p in train_df['path']]
exact = train_df[train_df.duplicated('md5', keep=False)].sort_values('md5')
print('Exact duplicate files:', len(exact))
R['exact_dups'] = int(len(exact))

if len(exact):
    conflict = exact.groupby('md5')['label'].nunique()
    print('Duplicate groups with conflicting labels:', int((conflict > 1).sum()))
    show = exact.groupby('md5').head(2).head(6)
    fig, axes = plt.subplots(1, len(show), figsize=(2.2 * len(show), 2.6))
    axes = np.atleast_1d(axes)
    for ax, (_, row) in zip(axes, show.iterrows()):
        ax.imshow(Image.open(row['path']).convert('RGB'))
        ax.set_title(f"{row['label']}\n{row['file']}", fontsize=7)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

### Task 2.2: Split by group (stratified)

In [ ]:
# Validation split BY GROUP: no group appears in both train and val
sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=42)
tr_idx, va_idx = next(sgkf.split(train_df, train_df['label'],
                                 train_df['group']))
tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
va_df = train_df.iloc[va_idx].reset_index(drop=True)
if not TEST_DIR:                                   # carve a grouped test set
    tr_idx, te_idx = next(sgkf.split(tr_df, tr_df['label'], tr_df['group']))
    tr_df, test_df = (tr_df.iloc[tr_idx].reset_index(drop=True),
                      tr_df.iloc[te_idx].reset_index(drop=True))

shared = set(tr_df.group) & set(va_df.group)
shared_te = set(tr_df.group) & set(test_df.group)
print(f'train {len(tr_df)} | val {len(va_df)} | test {len(test_df)}')
print('Groups shared by train and val:', len(shared))        # must be 0
print('Groups shared by train and test:', len(shared_te))    # must be 0
R['split'] = (len(tr_df), len(va_df), len(test_df))
R['shared_groups'] = len(shared) + len(shared_te)

split_table = pd.DataFrame({'train': tr_df.label.value_counts(),
                            'val': va_df.label.value_counts(),
                            'test': test_df.label.value_counts()}).reindex(classes)
print(split_table.to_string())

# Exact duplicates ACROSS the final splits
for a_name, a, b_name, b in [('train', tr_df, 'val', va_df),
                             ('train', tr_df, 'test', test_df),
                             ('val', va_df, 'test', test_df)]:
    print(f'Exact duplicate files {a_name} vs {b_name}:', len(set(a.md5) & set(b.md5)))
R['exact_cross'] = int(len(set(tr_df.md5) & set(va_df.md5)) + len(set(tr_df.md5) & set(test_df.md5))
                       + len(set(va_df.md5) & set(test_df.md5)))

### Task 2.3: Near-duplicates between the splits
The fingerprint is a standardized 32 × 32 grey thumbnail. Per the lab notes, recompressed copies score about 0.007 and genuinely different images 0.12 or more, so **0.10** is the flag threshold. I run it on GPU for speed and compare **test → train** and **val → train**.

In [ ]:
def fingerprint(path, n=32):
    """32 x 32 grayscale thumbnail, standardized: robust to resizing,
    recompression, and brightness changes."""
    with Image.open(path) as im:
        im.draft('L', (n * 4, n * 4))                      # fast JPEG decode
        a = np.asarray(im.convert('L').resize((n, n)), dtype=np.float32)
    return ((a - a.mean()) / (a.std() + 1e-6)).flatten()

def fp_matrix(df):
    return torch.tensor(np.stack([fingerprint(p) for p in df['path']]), device=device)

def min_dist(a, b, chunk=512):
    """Mean absolute difference to the nearest row of b, for every row of a."""
    d, idx = [], []
    for i in range(0, len(a), chunk):
        m = torch.cdist(a[i:i + chunk], b, p=1) / a.shape[1]
        v, j = m.min(1)
        d.append(v.cpu()); idx.append(j.cpu())
    return torch.cat(d).numpy(), torch.cat(idx).numpy()

NEAR_T = 0.10
F_tr, F_va, F_te = fp_matrix(tr_df), fp_matrix(va_df), fp_matrix(test_df)
d_te, j_te = min_dist(F_te, F_tr)
d_va, j_va = min_dist(F_va, F_tr)
near_te, near_va = d_te < NEAR_T, d_va < NEAR_T
print(f'Test images with a near-duplicate in train: {near_te.sum()} of {len(test_df)}')
print(f'Val images with a near-duplicate in train:  {near_va.sum()} of {len(va_df)}')
print('Closest test->train distances:', np.sort(d_te)[:5].round(3))
R['near_test'] = int(near_te.sum()); R['near_val'] = int(near_va.sum())
R['n_test_before'] = int(len(test_df))

In [ ]:
# Before deleting anything: look at the closest pairs side by side
order = np.argsort(d_te)[:4]
fig, axes = plt.subplots(2, 4, figsize=(10, 5.2))
for k, i in enumerate(order):
    axes[0, k].imshow(Image.open(test_df.path[i]).convert('RGB'))
    axes[0, k].set_title(f'TEST {test_df.label[i]}\nd={d_te[i]:.3f}', fontsize=8)
    axes[1, k].imshow(Image.open(tr_df.path[j_te[i]]).convert('RGB'))
    axes[1, k].set_title(f'TRAIN {tr_df.label[j_te[i]]}', fontsize=8)
    axes[0, k].axis('off'); axes[1, k].axis('off')
plt.suptitle('Closest test image (top) and its nearest train image (bottom)')
plt.tight_layout()
plt.show()

In [ ]:
# Decision cell. Set True ONLY if the flagged pairs above really are the same image.
REMOVE_NEAR_DUPES = False
if REMOVE_NEAR_DUPES and (near_te.sum() or near_va.sum()):
    test_df = test_df[~near_te].reset_index(drop=True)
    va_df = va_df[~near_va].reset_index(drop=True)
    print('Removed flagged images -> val', len(va_df), '| test', len(test_df))
else:
    print('No images removed. REMOVE_NEAR_DUPES =', REMOVE_NEAR_DUPES)
R['removed_near'] = bool(REMOVE_NEAR_DUPES)
R['split_final'] = (len(tr_df), len(va_df), len(test_df))

## ✍️ Write it: Part 2 (Leakage audit)
**Duplicates found.** MD5 found **___ exact duplicate files** in the full dataset, and **___** across my final train / val / test splits. The near-duplicate fingerprint (threshold 0.10) flagged **___ of ___ test images** and **___ of ___ validation images** as having a close match in train. I displayed the closest pairs side by side (figure above) and they **(are / are not)** the same image: ______________________. **What I did:** ______________________ (e.g. kept them because the pairs only look alike because land cover is uniform, or removed them from val and test with `REMOVE_NEAR_DUPES = True`).

**Images per group.** My dataset has **___ images per group** (1.00). EuroSAT file names carry no scene or patient id, so each tile is its own group and the "0 shared groups" check is **true but weak**: I printed 0 shared groups, but neighbouring tiles cut from the same satellite scene can still look alike.

**Random vs grouped split (two sentences).** If images from the same patient, plant, or scene appear in both train and test, the model can score well by recognising that specific subject instead of the condition, so a random split inflates the test score. In this dataset the effect is probably smaller than for chest X-rays because there is no explicit subject id, but I cannot rule out scene-level similarity, so I treat the test score as a likely **upper bound** and say so in the README.

# Part 3: Transforms and Imbalance (10 min)
### Task 3.1: Datasets and transforms
**Augmentation choices for satellite imagery.** A top-down view has **no fixed "up"**, so horizontal and vertical flips are realistic (unlike chest X-rays, where anatomy is not symmetric). I **removed** the 10° rotation from the template because rotating a 64-pixel tile creates black corners that the model could learn as an artificial cue. I **kept** a mild random crop (scale 0.8-1.0) and mild brightness/contrast jitter (0.15), which imitate small framing differences and season/haze variation without inventing new colours.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
to_rgb = [transforms.Grayscale(num_output_channels=3)] if GRAYSCALE else []

train_tf = transforms.Compose(to_rgb + [
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),      # top-down view: no fixed orientation
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose(to_rgb + [
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ImageTable(Dataset):
    def __init__(self, df, tf):
        self.paths = df['path'].tolist()
        self.y = df['label'].map({c: i for i, c in enumerate(classes)}).tolist()
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i])
        img = img.convert('L' if GRAYSCALE else 'RGB')
        return self.tf(img), self.y[i]

train_ds = ImageTable(tr_df, train_tf)
val_ds = ImageTable(va_df, eval_tf)
test_ds = ImageTable(test_df, eval_tf)
print(len(train_ds), len(val_ds), len(test_ds))

### Task 3.2: Class weights or balanced sampling

In [ ]:
n_c = np.bincount(train_ds.y, minlength=len(classes))
class_w = len(train_ds) / (len(classes) * n_c)           # N / (K * n_c)
print(dict(zip(classes, class_w.round(3))))
loss_w = torch.tensor(class_w, dtype=torch.float32, device=device)
R['weights'] = {c: round(float(w), 3) for c, w in zip(classes, class_w)}

# Arithmetic for the largest and smallest class
N, K = len(train_ds), len(classes)
for i in (int(n_c.argmax()), int(n_c.argmin())):
    print(f'{classes[i]}: N / (K x n_c) = {N} / ({K} x {n_c[i]}) = {N / (K * n_c[i]):.3f}')
R['w_example'] = [(classes[i], N, K, int(n_c[i]), round(N / (K * n_c[i]), 3))
                  for i in (int(n_c.argmax()), int(n_c.argmin()))]

USE_SAMPLER = False      # try True: balanced batches instead of weighted loss
if USE_SAMPLER:
    sample_w = class_w[train_ds.y]                        # P(i) ~ 1 / n_class(i)
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    train_dl = DataLoader(train_ds, 32, sampler=sampler, num_workers=2)
    loss_w = None
else:
    train_dl = DataLoader(train_ds, 32, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, 64, num_workers=2)
test_dl = DataLoader(test_ds, 64, num_workers=2)
R['imbalance_method'] = 'balanced sampler' if USE_SAMPLER else 'class-weighted cross-entropy'

## ✍️ Write it: Part 3 (Transforms and imbalance)
**Class weights and arithmetic.** I used the formula **weight = N / (K × n_c)**, where N is the number of training images, K = 10 classes and n_c the images in that class. Worked example from my output for the largest class (**___**): `N / (K × n_c) = ___ / (10 × ___) = ___`. For the smallest class (**___**): `___ / (10 × ___) = ___`. The weights are close to 1 (about 0.8 to 1.2) because the imbalance is mild, so the smaller classes get only a slightly bigger push in the loss. **Method used:** class-weighted cross-entropy (`USE_SAMPLER = False`). *(If you also ran `USE_SAMPLER = True`, add: macro-F1 was ___ with the sampler versus ___ with weights.)*

**Augmentations kept / removed and why.**
| Augmentation | Decision | Reason |
|---|---|---|
| Random resized crop (scale 0.8-1.0) | **Kept** | Simulates small framing differences between tiles; mild so land cover stays recognisable |
| Horizontal and vertical flip | **Added** | A satellite looks straight down, so there is no fixed orientation; flips are realistic here |
| Colour jitter (brightness, contrast 0.15) | **Kept, mild** | Imitates season, sun angle and haze. Stronger jitter would turn forest into a different class visually |
| Rotation 10° | **Removed** | Creates black corners on 64 px tiles that could become a fake cue |
| Hue / strong colour shifts | **Not used** | Colour carries real evidence (water, vegetation); large shifts would destroy it |

# Part 4: Build and Train (25 min)
### Task 4.1: Backbone with a new head
Phase 1 trains only the new 10-class head on top of frozen ImageNet features. Phase 2 unfreezes the last block (`layer4`) with a 10× smaller learning rate than the head.

In [ ]:
def build_model(name, n_classes):
    if name == 'resnet18':
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
        body, head, last = m, m.fc, m.layer4
    elif name == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
        body, head, last = m, m.fc, m.layer4
    elif name == 'efficientnet_b0':
        w = models.EfficientNet_B0_Weights.DEFAULT
        m = models.efficientnet_b0(weights=w)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
        body, head, last = m, m.classifier, m.features[-1]
    else:
        raise ValueError(name)
    for p in body.parameters():
        p.requires_grad = False
    for p in head.parameters():
        p.requires_grad = True
    return m.to(device), head, last

model, head, last_block = build_model(BACKBONE, len(classes))
print(BACKBONE, 'trainable:',
      sum(p.numel() for p in model.parameters() if p.requires_grad))

### Task 4.2: Two-phase training, selected by macro-F1

In [ ]:
history = []

def predict(model, loader):
    model.eval()
    probs, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            probs.append(F.softmax(model(xb.to(device)), 1).cpu())
            ys.append(yb)
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()

def fit(model, params, epochs, lr, phase):
    opt = torch.optim.Adam(params, lr=lr)
    best, best_state = -1, None
    for ep in range(epochs):
        model.train()
        t0 = time.time()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            loss = F.cross_entropy(model(xb), yb, weight=loss_w)
            opt.zero_grad()
            loss.backward()
            opt.step()
        p, y = predict(model, val_dl)
        macro = f1_score(y, p.argmax(1), average='macro')     # not accuracy
        acc = float((p.argmax(1) == y).mean())
        history.append({'phase': phase, 'epoch': ep + 1, 'val_macro_f1': macro,
                        'val_acc': acc, 'sec': time.time() - t0})
        if macro > best:
            best, best_state = macro, copy.deepcopy(model.state_dict())
        print(f'phase {phase} epoch {ep + 1}: val macro-F1 {macro:.3f} | acc {acc:.3f} '
              f'({time.time() - t0:.0f}s)')
    model.load_state_dict(best_state)
    return best, best_state

print('Phase 1: head only')
f1_p1, state_p1 = fit(model, head.parameters(), epochs=2, lr=1e-3, phase=1)
print('Phase 2: fine-tune the last block')
for p in last_block.parameters():
    p.requires_grad = True
f1_p2, state_p2 = fit(model, [{'params': last_block.parameters(), 'lr': 1e-4},
                              {'params': head.parameters(), 'lr': 1e-3}],
                      epochs=3, lr=1e-3, phase=2)

# Keep the best checkpoint over BOTH phases (selection by validation macro-F1)
if f1_p2 < f1_p1:
    model.load_state_dict(state_p1)
    print('Phase 1 checkpoint was better; restored it.')
R['p1'], R['p2'] = float(f1_p1), float(f1_p2)
R['best_val'] = float(max(f1_p1, f1_p2))
R['epoch_sec'] = float(np.mean([h['sec'] for h in history]))
print(f'Best val macro-F1: phase 1 {f1_p1:.3f} | phase 2 {f1_p2:.3f}')

In [ ]:
h = pd.DataFrame(history)
h['step'] = range(1, len(h) + 1)
plt.figure(figsize=(7, 3.5))
plt.plot(h.step, h.val_macro_f1, 'o-', label='val macro-F1', color='#00838f')
plt.plot(h.step, h.val_acc, 's--', label='val accuracy', color='#9e9e9e')
plt.axvline(2.5, color='k', ls=':', lw=1)
plt.text(1.0, plt.ylim()[0] + 0.005, 'phase 1\n(head)', fontsize=8)
plt.text(3.0, plt.ylim()[0] + 0.005, 'phase 2\n(fine-tune)', fontsize=8)
plt.xlabel('epoch (cumulative)'); plt.ylabel('score')
plt.title('Validation score by epoch')
plt.legend(); plt.tight_layout(); plt.show()

## ✍️ Write it: Part 4 (Training and selection)
**Validation macro-F1 per phase.** After **phase 1** (head only, 2 epochs) the best validation macro-F1 was **___**. After **phase 2** (last block unfrozen, 3 epochs) it was **___**.

**Did fine-tuning help?** **(Yes / No / Slightly)**: macro-F1 changed by **___** points. ______________________. *(Typical reading: ImageNet features are generic, but satellite textures differ from everyday photos, so letting `layer4` adapt usually helps most on the classes that look alike, such as the crop and vegetation classes. If your phase 2 score was lower, say so: the code then restores the phase 1 checkpoint, which is exactly why selection is by validation score.)*

**Selection metric.** The best checkpoint is chosen by **validation macro-F1**, not accuracy. With only mild imbalance the two curves are close (see plot), but macro-F1 is the safer choice because it gives every class equal weight, and in this project a missed class is a hole in a map.

**Time per epoch and extrapolation.** One epoch took about **___ seconds** on the Colab T4. With more time I would run about **___ more epochs** in phase 2 (for example 8-10 total), stop when validation macro-F1 stops improving for 2 epochs, and then try a learning-rate decay and the balanced sampler (`USE_SAMPLER = True`) to compare.

# Part 5: Test Once, and Set the Threshold (15 min)
The test set is used **once** for the final report. All decisions (checkpoint, threshold) were made on validation data.
### Task 5.1: Test-set report

In [ ]:
p_test, y_test = predict(model, test_dl)
pred = p_test.argmax(1)
print(classification_report(y_test, pred, target_names=classes, digits=3))
macro_test = f1_score(y_test, pred, average='macro')
print('Macro-F1:', round(macro_test, 3))
R['macro_test'] = float(macro_test)
R['acc_test'] = float((pred == y_test).mean())

rep = classification_report(y_test, pred, target_names=classes, digits=3, output_dict=True)
recall = pd.Series({c: rep[c]['recall'] for c in classes}).sort_values()
R['recall'] = recall.to_dict()
R['weakest'] = (recall.index[0], float(recall.iloc[0]))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
recall.plot(kind='barh', ax=ax[0], color='#00838f')
ax[0].axvline(0.90, color='r', ls='--', lw=1, label='0.90 reference line')
ax[0].set_xlim(max(0, recall.min() - 0.05), 1.0)
ax[0].set_title('Per-class recall (test, used once)')
ax[0].legend()
cm = confusion_matrix(y_test, pred)
ConfusionMatrixDisplay(cm, display_labels=classes).plot(cmap='Blues', ax=ax[1],
                                                        xticks_rotation=60, colorbar=False)
ax[1].set_title('Confusion matrix (counts)')
plt.tight_layout()
plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
a, b = np.unravel_index(off.argmax(), off.shape)
R['confused'] = (classes[a], classes[b], int(off[a, b]))
print(f'Weakest class by recall: {R["weakest"][0]} ({R["weakest"][1]:.3f})')
print(f'Most confused pair: true {classes[a]} predicted as {classes[b]} ({off[a, b]} images)')
for c in ('River', 'SeaLake'):
    if c in classes:
        print(f'{c}: recall {rep[c]["recall"]:.3f} | precision {rep[c]["precision"]:.3f}')

### Task 5.2: Threshold for a target recall (flood-relevant water class)
The binary threshold cell of the lab needs a single positive class. This is a 10-class task, so I run it as a **one-vs-rest analysis for `River`**, the water class most relevant to flood mapping. The threshold is chosen on **validation** data for recall ≥ 0.95 and applied once to the test set. The PPV at a "realistic prevalence" uses an **assumed** value that I state explicitly, because EuroSAT tiles were sampled per class and do not reflect how common rivers are in a real region.

In [ ]:
POS = POSITIVE if (len(classes) == 2 and POSITIVE) else 'River'
TR = TARGET_RECALL or 0.95
ASSUMED_PREV = 0.05          # ASSUMPTION: share of tiles in a real region that are river. Edit and justify.

pos = classes.index(POS)
p_val, y_val = predict(model, val_dl)
fpr, tpr, thr = roc_curve(y_val == pos, p_val[:, pos])
t = thr[np.argmax(tpr >= TR)]                        # highest t reaching target
print(f'Threshold chosen on VALIDATION for {POS} recall >= {TR}: {t:.3f}')

s = p_test[:, pos]
yp, yt = s >= t, y_test == pos
tp, fn = (yp & yt).sum(), (~yp & yt).sum()
tn, fp = (~yp & ~yt).sum(), (yp & ~yt).sum()
sens, spec = tp / (tp + fn), tn / (tn + fp)
ppv_test = tp / max(tp + fp, 1)
prev_test = yt.mean()
ppv_real = sens * ASSUMED_PREV / (sens * ASSUMED_PREV + (1 - spec) * (1 - ASSUMED_PREV))
print(f'Test AUC {roc_auc_score(yt, s):.3f}')
print(f'Sensitivity {sens:.3f}   Specificity {spec:.3f}   PPV {ppv_test:.3f}  (test prevalence {prev_test:.3f})')
print('\nPPV at an assumed realistic prevalence (Bayes, working shown):')
print(f'PPV = sens*prev / (sens*prev + (1-spec)*(1-prev))')
print(f'    = {sens:.3f}*{ASSUMED_PREV} / ({sens:.3f}*{ASSUMED_PREV} + {1 - spec:.3f}*{1 - ASSUMED_PREV})')
print(f'    = {ppv_real:.3f}')
R['thr'] = (POS, float(t), float(sens), float(spec), float(ppv_test), float(prev_test),
            float(ppv_real), ASSUMED_PREV, float(roc_auc_score(yt, s)), TR)

### Task 5.3: Season / haze stress test (the known pitfall)
I cannot download images from other regions or seasons here, so this is a **proxy experiment**: I apply realistic appearance shifts (brightness, contrast, desaturation, haze, blur) to the **validation** images and measure how much macro-F1 drops. I use validation, **not the test set**, so the test set stays "used once". This does **not** replace real out-of-region testing, but it shows how fragile the model is to appearance changes.

In [ ]:
class Shifted(Dataset):
    def __init__(self, df, fn):
        self.paths, self.fn = df['path'].tolist(), fn
        self.y = df['label'].map({c: i for i, c in enumerate(classes)}).tolist()
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return eval_tf(self.fn(img)), self.y[i]

def haze(img, a=0.30):
    return Image.blend(img, Image.new('RGB', img.size, (235, 235, 235)), a)

shifts = {
    'original': lambda im: im,
    'darker (low sun / winter)': lambda im: TF.adjust_brightness(im, 0.6),
    'brighter (summer glare)': lambda im: TF.adjust_brightness(im, 1.4),
    'low contrast': lambda im: TF.adjust_contrast(im, 0.5),
    'faded colour (season)': lambda im: TF.adjust_saturation(im, 0.4),
    'haze / thin cloud': haze,
    'blur (coarser sensor)': lambda im: im.filter(ImageFilter.GaussianBlur(1.5)),
}
stress = {}
for name, fn in shifts.items():
    dl = DataLoader(Shifted(va_df, fn), 64, num_workers=2)
    pp, yy = predict(model, dl)
    stress[name] = f1_score(yy, pp.argmax(1), average='macro')
    print(f'{name:28s} val macro-F1 {stress[name]:.3f}')
R['stress'] = stress

s_ser = pd.Series(stress)
plt.figure(figsize=(8, 3.8))
s_ser.plot(kind='barh', color=['#00838f'] + ['#ef6c00'] * (len(s_ser) - 1))
plt.xlim(max(0, s_ser.min() - 0.05), 1.0)
plt.xlabel('validation macro-F1'); plt.title('Appearance-shift stress test (proxy for season / haze)')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## ✍️ Write it: Part 5 (Evaluation)
**Overall result (test set, used once).** Macro-F1 = **___**, accuracy = **___**. The **weakest class is ___ with recall ___**. The most confused pair is **true ___ predicted as ___ (___ images)**. *(Typical for EuroSAT: crop and vegetation classes such as AnnualCrop, PermanentCrop, Pasture and HerbaceousVegetation look alike from above, so they confuse each other, while Forest, SeaLake and Residential are easy. Edit this sentence to match your own confusion matrix.)* In a land-use map this error type matters: confusing Pasture with crops changes crop-area estimates even though both are "green".

**Flood-relevant water classes.** River recall = **___** and precision = **___**; SeaLake recall = **___**, precision = **___**. River is harder than SeaLake because narrow rivers cover a small part of the tile and resemble roads or crop edges.

**Threshold analysis (one-vs-rest, River).** On validation data I chose the threshold **___** for target recall **0.95**. On the test set this gave **sensitivity ___, specificity ___, PPV ___** at the test prevalence (___). **Target recall reached on test? (Yes / No)**: ______________________ *(a threshold set on validation does not guarantee the same recall on a new set; if it is slightly below 0.95 on test, report that honestly).*

**PPV at two prevalences, in plain language.** At the test prevalence, PPV is **___**. At my assumed realistic prevalence of **___** (an assumption, not a measurement), the **same classifier** has PPV **___**. Working: `PPV = (sens × prev) / (sens × prev + (1 − spec) × (1 − prev))`. PPV falls when the target class is rare because the many non-river tiles produce more false alarms than the few real river tiles produce true hits, even with unchanged sensitivity and specificity. For flood mapping this means many predicted "river" tiles would need a human check.

**Stress test (region / season pitfall).** Original validation macro-F1 was **___**. The largest drop came from **___ (___)**, a fall of **___** points; the mildest shift was **___**. *(Interpretation: if colour-style shifts hurt most, the model leans on colour more than on texture and shape; this links to my Part 1 shortcut suspect. Real out-of-region testing would be harsher.)* **Conclusion:** the test score describes European tiles under the conditions in this dataset only; it should **not** be quoted as expected accuracy in other regions or seasons.

# Part 6: Grad-CAM Shortcut Check (15 min)
Grad-CAM weights the last block's feature maps by the average gradient of the class score and keeps the positive evidence (ReLU of the weighted sum). The heatmap shows **where** the model looked. I check six predictions (three right, three wrong), then add two numeric tests: how much of the heat sits on the **tile border**, and how performance changes when the border or the centre is masked.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(self._save_act)
        layer.register_full_backward_hook(self._save_grad)
    def _save_act(self, m, i, o):
        self.acts = o.detach()
    def _save_grad(self, m, gi, go):
        self.grads = go[0].detach()
    def __call__(self, x, cls=None):
        self.model.eval()
        x = x.unsqueeze(0).to(device).requires_grad_(True)
        logits = self.model(x)
        cls = int(logits.argmax()) if cls is None else cls
        self.model.zero_grad()
        logits[0, cls].backward()
        alpha = self.grads.mean(dim=(2, 3), keepdim=True)        # (1/Z) sum dy/dA
        cam = F.relu((alpha * self.acts).sum(1)).squeeze()       # ReLU(sum a*A)
        cam = F.interpolate(cam[None, None], size=x.shape[-2:],
                            mode='bilinear').squeeze()
        return (cam / (cam.max() + 1e-8)).cpu().numpy(), cls

for p in model.parameters():
    p.requires_grad = True
cam = GradCAM(model, last_block)

def first_right(cname):
    if cname not in classes:
        return None
    ix = np.where((pred == y_test) & (y_test == classes.index(cname)))[0]
    return int(ix[0]) if len(ix) else None

right = [first_right('River'), first_right('SeaLake'), first_right(R['weakest'][0])]
right = list(dict.fromkeys(i for i in right if i is not None))
for i in np.where(pred == y_test)[0]:
    if len(right) >= 3:
        break
    if int(i) not in right:
        right.append(int(i))
wrong = np.where(pred != y_test)[0][:3].tolist()

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax in axes.flat:
    ax.axis('off')
for ax, i in zip(axes.flat, right + wrong):
    x, y = test_ds[i]
    heat, c = cam(x)
    img = Image.open(test_ds.paths[i]).convert('RGB').resize((IMG_SIZE,) * 2)
    ax.imshow(img)
    ax.imshow(heat, cmap='jet', alpha=0.4)
    tag = 'RIGHT' if i in right else 'WRONG'
    ax.set_title(f'{tag}: true {classes[y]} | pred {classes[c]}', fontsize=9)
plt.tight_layout()
plt.show()
R['n_wrong_shown'] = len(wrong)

In [ ]:
# Numeric check 1: how much Grad-CAM mass sits in the outer frame? (uniform heat would give `expected`)
m = IMG_SIZE // 10
def border_share(heat):
    return 1 - heat[m:-m, m:-m].sum() / (heat.sum() + 1e-8)

rng = np.random.RandomState(0)
sel = rng.choice(len(test_ds), min(200, len(test_ds)), replace=False)
shares = [border_share(cam(test_ds[int(i)][0])[0]) for i in sel]
expected = 1 - ((IMG_SIZE - 2 * m) / IMG_SIZE) ** 2
ratio = float(np.mean(shares) / expected)
print(f'Border (outer {m}px frame) share of heat: {np.mean(shares):.3f} | expected if uniform: {expected:.3f} | ratio {ratio:.2f}')

# Numeric check 2: masking experiment on VALIDATION data
class Masked(Dataset):
    def __init__(self, df, mode):
        self.base, self.mode = ImageTable(df, eval_tf), mode
        self.y = self.base.y
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        k = IMG_SIZE // 5
        keep = torch.zeros(IMG_SIZE, IMG_SIZE)
        if self.mode == 'centre_only':
            keep[k:-k, k:-k] = 1
        elif self.mode == 'border_only':
            keep[:] = 1; keep[k:-k, k:-k] = 0
        else:
            keep[:] = 1
        return x * keep, y

mask_res = {}
for mode in ('full', 'centre_only', 'border_only'):
    pp, yy = predict(model, DataLoader(Masked(va_df, mode), 64, num_workers=2))
    mask_res[mode] = f1_score(yy, pp.argmax(1), average='macro')
    print(f'{mode:12s} val macro-F1 {mask_res[mode]:.3f}')
R['border_ratio'] = ratio; R['mask'] = mask_res

R['verdict'] = ('shortcut suspected (heat concentrated on the tile border)' if ratio > 1.3
                else 'no shortcut found in the checks run (colour cast not fully excluded)')
print('\nAuto-suggested verdict:', R['verdict'])
print('Confirm or edit this by LOOKING at the six heatmaps before you submit.')

## ✍️ Write it: Part 6 (Grad-CAM verdict)
*(Look at your own six heatmaps and replace each line with what you actually see. The lines below show the format and the kind of reasoning expected.)*

| # | Case | Where the model looked | Legitimate evidence? |
|---|---|---|---|
| 1 | RIGHT: River | ______ (e.g. along the dark winding water channel) | ______ (Yes: the water body itself) |
| 2 | RIGHT: SeaLake | ______ (e.g. spread over the uniform blue area) | ______ (Yes, but colour alone could explain it) |
| 3 | RIGHT: ______ | ______ | ______ |
| 4 | WRONG: true ___ → pred ___ | ______ (e.g. on a field boundary or road, not the dominant cover) | ______ (Partly: the cue is real but ambiguous) |
| 5 | WRONG: true ___ → pred ___ | ______ | ______ |
| 6 | WRONG: true ___ → pred ___ | ______ | ______ |

**Numeric checks.** The share of Grad-CAM heat in the outer border frame was **___** against **___** expected for uniform attention (ratio **___**). Validation macro-F1 was **___** on full tiles, **___** with only the centre kept, and **___** with only the border kept. *(If both partial versions keep a reasonable score, the model uses evidence spread over the whole tile, which is what land-cover texture should look like. A very high border-only score, or a ratio far above 1, would point to a border or edge artefact.)*

**Overall verdict: ______________ (no shortcut found / shortcut suspected / shortcut confirmed).** ______________________ *(Be precise: heatmaps show where the model looked, not why. Colour is still a possible shortcut that Grad-CAM cannot fully separate from real water or vegetation evidence.)*

**Follow-up experiment.** To test the colour-shortcut suspect from Part 1, I would (1) convert the tiles to grayscale (or randomly shuffle colour channels) and re-score: if macro-F1 collapses for classes like Forest or SeaLake, the model leans on colour; (2) crop out the highlighted region and re-score to see whether the prediction depends on it; (3) evaluate on tiles from another region or season, which is the only test that directly addresses the region/season pitfall.

# Part 7: Save and Document
### Save the model

In [ ]:
torch.save({'state_dict': model.state_dict(), 'classes': classes,
            'backbone': BACKBONE, 'img_size': IMG_SIZE,
            'grayscale': GRAYSCALE}, '/content/week6_model.pt')
print('Saved week6_model.pt')

### Auto-filled README (datasheet + model card)
The next cell writes `README.md` using the **real numbers from your own run** (stored in the `R` dictionary). Read it, edit the sentences that do not match your results, and make sure the wording is your own.

In [ ]:
c = R
big = max(c['counts'], key=c['counts'].get); small = min(c['counts'], key=c['counts'].get)
thr_cls, thr_t, sens, spec, ppv_t, prev_t, ppv_r, aprev, auc, trg = c['thr']
w_txt = ', '.join(f'{k} {v}' for k, v in c['weights'].items())
st = c['stress']; worst = min((k for k in st if k != 'original'), key=st.get)

readme = f"""## Week 6: Land-use classification of Sentinel-2 tiles (EuroSAT) with {BACKBONE}

**Who benefits:** land-use planners, agriculture and flood-risk analysts in Europe-like settings. **Why it matters:** land-use and crop maps feed flood-risk planning and crop monitoring; manual surveys are slow and costly, and wrong maps misallocate effort and money.

**Dataset:** EuroSAT (Sentinel-2 RGB, Helber et al.), downloaded with torchvision (RGB version); {c['n_images']} images, {c['k']} classes; labels from Sentinel-2 scenes with the original authors' curation. License: check the license in the authors' repository (github.com/phelber/EuroSAT) and cite it here: ____________.

**Data audit:** imbalance ratio {c['ratio']:.1f} (largest {big} {c['counts'][big]}, smallest {small} {c['counts'][small]}); exact duplicates {c['exact_dups']} (across splits: {c['exact_cross']}); test images with a near-duplicate in train {c['near_test']} of {c['n_test_before']} (val: {c['near_val']}); stratified split with no shared groups (every image is its own group, so scene-level similarity is not excluded). Final split train/val/test = {c['split_final']}.

**Imbalance handling:** {c['imbalance_method']}, weights [{w_txt}].

**Results (test, used once):** macro-F1 {c['macro_test']:.3f}, accuracy {c['acc_test']:.3f}; weakest class {c['weakest'][0]} (recall {c['weakest'][1]:.3f}); most confused: true {c['confused'][0]} predicted as {c['confused'][1]} ({c['confused'][2]} images). One-vs-rest for {thr_cls}: threshold {thr_t:.3f} (set on val for recall >= {trg}); test sensitivity {sens:.3f}, specificity {spec:.3f}, PPV {ppv_t:.3f} at test prevalence {prev_t:.3f} and {ppv_r:.3f} at an assumed prevalence of {aprev}.

**Grad-CAM:** border share of heat {c['border_ratio']:.2f}x the uniform expectation; masking test (val macro-F1) full {c['mask']['full']:.3f}, centre-only {c['mask']['centre_only']:.3f}, border-only {c['mask']['border_only']:.3f}. Verdict: {c['verdict']}. Colour as a shortcut was not fully excluded.

**Stress test (proxy for season/haze, on validation):** original {st['original']:.3f}; worst shift "{worst}" {st[worst]:.3f}.

**Limitations:** European Sentinel-2 tiles only, 64 x 64 px at coarse resolution, limited acquisition conditions; no flood class, so the model does **not** detect floods. Not for operational flood response, insurance, or subsidy decisions without human review and local validation. Test score is likely an upper bound because scene-level near-neighbours were not excluded.

### Responsibility questions (slide 34)
1. **Who could be harmed and how?** Planners, farmers or insurers who rely on wrong maps: confusing Pasture with crops or River with road changes area estimates, subsidies and flood-exposure numbers; regions not represented in the data are served worst.
2. **Where was the model not validated?** Outside Europe, other seasons and crop calendars, cloud or haze, other sensors or resolutions, larger tiles, and actual flood scenes.
3. **What is the human role?** The model proposes a map; an analyst reviews low-confidence tiles and the weakest classes, and no automatic decision on compensation, evacuation or funding is made from the output.
4. **What evidence would change my conclusions?** Real out-of-region / out-of-season test results, a confirmed colour or border shortcut, near-duplicate leakage found between splits, or a label audit showing noisy labels.
"""
open('/content/README.md', 'w').write(readme)
display(Markdown(readme))

### LinkedIn post (auto-filled, then edit in your own words)

In [ ]:
post = f"""Week 6 of Applied AI: I picked my own problem.

Land-use and crop maps drive flood-risk planning and crop monitoring, and satellites produce far more tiles than people can label by hand.
I fine-tuned {BACKBONE} on {c['n_images']} Sentinel-2 tiles (EuroSAT, {c['k']} classes) and reached macro-F1 {c['macro_test']:.2f}.

What surprised me: the weakest class was {c['weakest'][0]} (recall {c['weakest'][1]:.2f}), mostly confused with similar green land cover, and a simple haze / season-style change to the images dropped validation macro-F1 from {st['original']:.2f} to {st[worst]:.2f}. Accuracy alone would have hidden it, and it is a reminder that a model trained on one region and season may not transfer.

[Grad-CAM image]
#ComputerVision #DeepLearning #RemoteSensing #LearningInPublic"""
print(post)

### Download your files from Colab
Colab deletes `/content` when the session ends. Run this cell to download the model and the README before you close the tab.

In [ ]:
try:
    from google.colab import files
    files.download('/content/README.md')
    files.download('/content/week6_model.pt')
except Exception as e:
    print('Download skipped (not running in Colab):', e)

# Check Your Understanding
**1. Why does a random split of chest X-rays overestimate performance?** One patient contributes several X-rays that look very similar. A random split puts some of a patient's images in training and others in test, so the model can score well by recognising that patient's anatomy, scanner and body features instead of pneumonia. The test then measures memory of seen patients, not performance on **new** patients. A group split keeps every patient on one side only.

**2. Two classes with 900 and 100 training images. Class weights with N / (K × n_c).** N = 1000, K = 2.
- Majority class: 1000 / (2 × 900) = 1000 / 1800 = **0.556**
- Minority class: 1000 / (2 × 100) = 1000 / 200 = **5.0**

The minority class counts 9× more per example (5.0 / 0.556 = 9), which equals the inverse of the class-size ratio, so both classes contribute equal total weight to the loss (900 × 0.556 = 100 × 5.0 = 500).

**3. Focal loss, γ = 2, p_t = 0.8: down-weighting factor?** The modulating factor is (1 − p_t)^γ = (1 − 0.8)² = 0.2² = **0.04**. The example's loss is multiplied by 0.04, i.e. down-weighted **25 times** (1 / 0.04), so easy examples contribute little and the model focuses on hard ones.

**4. Sensitivity 0.90, specificity 0.95, prevalence 2%: PPV.** Per 10,000 people: 200 have the condition and 9,800 do not.
- True positives = 0.90 × 200 = 180
- False positives = (1 − 0.95) × 9,800 = 0.05 × 9,800 = 490
- PPV = 180 / (180 + 490) = 180 / 670 = **0.269 (about 27%)**

So fewer than 3 in 10 positive results are real, even though the test looks accurate, because the condition is rare.

**5. When is macro-F1 a better selection metric than weighted F1?** Macro-F1 averages the F1 of every class equally; weighted F1 averages by class size, so large classes dominate. Macro-F1 is better when **every class matters equally, or the rare class is the important one** (a disease, a flood, a defect), because a model that ignores a small class can still have a high weighted F1 and accuracy but cannot get a high macro-F1.

**6. Grad-CAM highlights a hospital label in the corner of X-rays. What next?** This is a **shortcut**: the model reads the label (which hospital, and so how sick the patients tend to be) instead of the lungs. Next steps: (1) confirm with a test: mask or crop the corner and re-score; if performance drops sharply, the shortcut is confirmed; (2) remove or cover the label region in all splits (crop or mask), or train on cleaned images; (3) retrain and re-check Grad-CAM; (4) evaluate on external data from another hospital; (5) report the issue in the datasheet and do not use the earlier score.

# Deliverables Checklist
- [x] Colab setup: GPU check and EuroSAT download (Part -1)
- [x] Configuration cell set for EuroSAT (auto-detects the data path)
- [x] Class counts, imbalance ratio, image sizes and modes (Part 1)
- [x] Exact and near-duplicate counts, flagged pairs inspected (Part 2)
- [x] Group split with 0 shared groups, with the limitation of single-image groups explained
- [x] Class weights with arithmetic shown (Part 3)
- [x] Best epoch chosen by validation macro-F1, over both phases (Part 4)
- [x] Test evaluated once: report, per-class recall, confusion matrix (Part 5)
- [x] Threshold from validation, sensitivity / specificity / PPV at two prevalences (one-vs-rest River)
- [x] Season / haze stress test for the region-and-season pitfall
- [x] Grad-CAM on right and wrong predictions with a verdict and follow-up experiment (Part 6)
- [x] A "Write it" markdown cell after every part
- [x] README with datasheet, model card and the four responsibility answers (Part 7)
- [x] Check Your Understanding answers
- [ ] LinkedIn post (copy, edit, and publish by the end of the week)